In [ ]:
# Standardbibliothek
from pathlib import Path
from importlib.metadata import version
import sys
import xml.etree.ElementTree as ET

# Daten und Video
import numpy as np
import pandas as pd
import cv2

# Darstellung
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
from IPython.display import display, Markdown

In [ ]:
PROJECT_DIR = Path(
    "/Volumes/T7/More Than A Hack 2027/Initial Challenge"
)

METADATA = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.xml"
RAW = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.txt"
TAGS = PROJECT_DIR / "tactical/BAR-ATH_tactical_smart_tagging.xml"

OUT = PROJECT_DIR / "output/ball_evidence"
OUT.mkdir(parents=True, exist_ok=True)

print("Projektordner erreichbar:", PROJECT_DIR.is_dir())
print("Metadaten vorhanden:", METADATA.is_file())
print("Tracking vorhanden:", RAW.is_file())
print("Smart Tags vorhanden:", TAGS.is_file())

In [ ]:
# XML-Metadaten einlesen
root = ET.parse(METADATA).getroot()

# Informationen zum Trackingformat
spec = root.find("./DataFormatSpecifications/DataFormatSpecification")

# Mannschaften auslesen
teams = root.findall("./Metadata/Teams/Team")

# Tracking-Kanäle auslesen
channels = root.findall("./Metadata/PlayerChannels/PlayerChannel")

# Ergebnisse anzeigen
print("Startframe:", spec.get("startFrame"))
print("Endframe:", spec.get("endFrame"))
print("Bildrate:", root.findtext("./Metadata/GlobalConfig/FrameRate"))

print("\nMannschaften:")
for team in teams:
    print(team.get("id"), team.findtext("Name"))

print("\nAnzahl Spielerkanäle:", len(channels))
print("Anzahl Tracking-Slots:", len(spec.findall("./SplitRegister/SplitRegister")))

In [ ]:
# Trackingkanal -> Spieler-/Track-ID
channel_to_player = {
    c.get("id"): c.get("playerId")
    for c in channels
}

# Spieler-/Track-ID -> Mannschaft
player_to_team = {
    p.get("id"): p.get("teamId")
    for p in root.findall("./Metadata/Players/Player")
}

team_ids = [team.get("id") for team in teams]

# Slots den Mannschaften zuordnen
slot_teams = []

for group in spec.findall("./SplitRegister/SplitRegister"):
    channel_id = group.find("PlayerChannelRef").get("playerChannelId")
    player_id = channel_to_player[channel_id]
    team_id = player_to_team[player_id]

    side = "H" if team_id == team_ids[0] else (
        "A" if team_id == team_ids[1] else "?"
    )
    slot_teams.append(side)

print("Barcelona-Slots:", slot_teams.count("H"))
print("Athletic-Slots:", slot_teams.count("A"))
print("Unbekannte Slots:", slot_teams.count("?"))
print("Erste 12 Slot-Zuordnungen:", list(enumerate(slot_teams[:12])))


In [ ]:
# Erste Trackingzeile einlesen
with RAW.open(encoding="utf-8") as file:
    line = file.readline().strip()

# Frame-ID, Spieler und Ball trennen
frame_id, players_raw, ball_raw = line.split(":")

# Positionswerte in ein NumPy-Array umwandeln
values = np.fromstring(
    players_raw.replace(";", ",") + "," + ball_raw,
    sep=",",
    dtype=np.float32
).reshape(-1, 3)

# Spieler und Ball separieren
players = values[:-1]
ball = values[-1]

# Prüfen, welche Positionen beobachtet wurden
observed = np.isfinite(players[:, :2]).all(axis=1)

print("Tracking-Frame:", frame_id)
print("Anzahl Spieler-Slots:", len(players))
print("Beobachtete Spieler-Slots:", observed.sum())
print("Ballposition vorhanden:", np.isfinite(ball[:2]).all())
print("Array-Form:", values.shape)

In [ ]:
team_mask = np.array(slot_teams)
records = []

with RAW.open(encoding="utf-8") as file:
    for _, line in zip(range(1000), file):

        frame_id, players_raw, ball_raw = line.strip().split(":")

        values = np.fromstring(
            players_raw.replace(";", ",") + "," + ball_raw,
            sep=",",
            dtype=np.float32
        )

        if values.size != 47 * 3:
            raise ValueError(f"Ungültige Daten in Frame {frame_id}")

        positions = values.reshape(47, 3)

        player_valid = np.isfinite(
            positions[:-1, :2]
        ).all(axis=1)

        ball_valid = np.isfinite(
            positions[-1, :2]
        ).all()

        records.append({
            "frame": int(frame_id),
            "n_H": int(player_valid[team_mask == "H"].sum()),
            "n_A": int(player_valid[team_mask == "A"].sum()),
            "ball_valid": bool(ball_valid)
        })

df_sample = pd.DataFrame(records)

print("Beobachtete Spieler pro Frame:")
print(df_sample[["n_H", "n_A"]].describe().round(2))

print("\nBallverfügbarkeit:")
print(f"{df_sample['ball_valid'].mean():.1%}")

print("\nErste 10 Frames:")
display(df_sample.head(10))

In [ ]:
# Taktisches Video automatisch finden
video_path = "/Volumes/T7/More Than A Hack 2027/Initial Challenge/tactical/BAR-ATH_tactical.mp4"

cap = cv2.VideoCapture(str(video_path))
assert cap.isOpened(), "Video konnte nicht geöffnet werden."

# Video-Frames (0-basiert)
frames_to_show = [0, 249, 499, 749, 11188, 50000]

fig, axes = plt.subplots(6, 1, figsize=(16, 25))

for ax, frame in zip(axes, frames_to_show):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame)
    success, image = cap.read()

    if success:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        ax.imshow(image)
        ax.set_title(f"Frame {frame} – {frame/25:.1f} Sekunden")
    else:
        ax.set_title(f"Frame {frame}: nicht lesbar")

    ax.axis("off")

cap.release()
plt.tight_layout()
plt.show()

In [ ]:

import xml.etree.ElementTree as ET

tag_root = ET.parse(TAGS).getroot()

def show_structure(element, depth=0, max_depth=3):
    print(
        "  " * depth,
        element.tag,
        element.attrib,
        (element.text or "").strip()[:60]
    )

    if depth < max_depth:
        for child in list(element)[:3]:
            show_structure(child, depth + 1, max_depth)

show_structure(tag_root)

# Alle verfügbaren Smart-Tag-Kategorien
rows = tag_root.findall("./ROWS/row")

categories = sorted({
    row.findtext("code", "").strip()
    for row in rows
})

print(f"Anzahl Kategorien: {len(categories)}\n")

for category in categories:
    print(category)


In [ ]:

# SET PIECES in den Smart Tags finden
set_pieces_row = next(
    row for row in rows
    if row.findtext("code", "").strip() == "SET PIECES"
)

# XML-Struktur der Kategorie untersuchen
print("Elemente der Kategorie SET PIECES:\n")

for child in set_pieces_row:
    print(
        child.tag,
        "| Attribute:", child.attrib,
        "| Unterelemente:", len(child)
    )

print("\nXML-Ausschnitt:\n")
print(
    ET.tostring(
        set_pieces_row,
        encoding="unicode"
    )[:2000]
)


# Alle erkannten Ereignisse auslesen
instances = tag_root.findall("./ALL_INSTANCES/instance")

print("Anzahl Ereignisse:", len(instances))

# Die ersten drei Ereignisse vollständig anzeigen
for i, instance in enumerate(instances[:3], start=1):
    print(f"\n--- Ereignis {i} ---")
    print(ET.tostring(instance, encoding="unicode"))


In [ ]:
kickoffs = []

for instance in instances:

    labels = {
        label.findtext("group"): label.findtext("text")
        for label in instance.findall("label")
    }

    event_type = labels.get("Type", "")

    if "Kick Off" in event_type:
        start = float(instance.findtext("start"))

        kickoffs.append({
            "type": event_type,
            "half": labels.get("Half"),
            "time_s": start,
            "video_frame": round(start * 25)
        })

df_kickoffs = pd.DataFrame(kickoffs)
display(df_kickoffs)

In [ ]:

# Frames rund um den ersten Anstoß
kickoff_frame = 11097

frames_to_show = kickoff_frame + np.array(
    [-50, 0, 25, 50, 100, 125]
)

cap = cv2.VideoCapture(str(video_path))

fig, axes = plt.subplots(3, 2, figsize=(15, 10))

for ax, frame in zip(axes.flat, frames_to_show):
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame))
    success, image = cap.read()

    if success:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        ax.imshow(image)

    ax.set_title(
        f"Frame {frame} | {frame / 25:.2f} s"
    )
    ax.axis("off")

cap.release()
plt.tight_layout()
plt.show()


In [ ]:

# Untersuchungsfenster um den ersten Anstoß
start_frame = 11050
end_frame = 11250

records = []

with RAW.open(encoding="utf-8") as file:
    for line in file:
        parts = line.strip().split(":")
        video_frame = int(parts[0]) - 1

        if video_frame < start_frame:
            continue
        if video_frame > end_frame:
            break

        ball = np.fromstring(
            parts[2], sep=",", dtype=np.float32
        )

        records.append({
            "frame": video_frame,
            "ball_x": ball[0],
            "ball_y": ball[1]
        })

df_ball = pd.DataFrame(records)

valid = np.isfinite(
    df_ball[["ball_x", "ball_y"]]
).all(axis=1)

print("Untersuchte Frames:", len(df_ball))
print("Frames mit Balltracking:", valid.sum())
print("Ballverfügbarkeit:", f"{valid.mean():.1%}")

print("\nErste Frames mit erkanntem Ball:")
display(df_ball.loc[valid].head(15))

In [ ]:

# Nur Frames mit gültiger Ballposition
ball_motion = df_ball.dropna(
    subset=["ball_x", "ball_y"]
).copy()

# Abstand zwischen aufeinanderfolgenden Beobachtungen
ball_motion["frame_gap"] = ball_motion["frame"].diff()

# Positionsänderung in Metern
ball_motion["dx_m"] = ball_motion["ball_x"].diff() * 105.3
ball_motion["dy_m"] = ball_motion["ball_y"].diff() * 68.0

# Ballgeschwindigkeit in m/s
ball_motion["speed_mps"] = (
    np.hypot(ball_motion["dx_m"], ball_motion["dy_m"])
    * 25 / ball_motion["frame_gap"]
)

# Nur direkte Folgeframes berücksichtigen
ball_motion.loc[
    ball_motion["frame_gap"] != 1, "speed_mps"
] = np.nan

display(
    ball_motion[
        ["frame", "ball_x", "ball_y", "speed_mps"]
    ].head(20).round(2)
)

In [ ]:

target_frame = 11187

# Gewünschten Frame aus den Trackingdaten lesen
with RAW.open(encoding="utf-8") as file:
    for line in file:
        frame_id, players_raw, ball_raw = line.strip().split(":")

        if int(frame_id) - 1 == target_frame:
            break
    else:
        raise ValueError("Frame nicht gefunden")

# Positionen einlesen
values = np.fromstring(
    players_raw.replace(";", ",") + "," + ball_raw,
    sep=",",
    dtype=np.float32
).reshape(47, 3)

player_xy = values[:-1, :2]
ball_xy = values[-1, :2]

# Gültige Positionen
valid = np.isfinite(player_xy).all(axis=1)

# Entfernung aller Spieler zum Ball in Metern
pitch_scale = np.array([105.3, 68.0])

distances = np.linalg.norm(
    (player_xy - ball_xy) * pitch_scale,
    axis=1
)

distances[~valid] = np.inf

# Nächsten Spieler je Team bestimmen
for team in ["H", "A"]:
    mask = (np.array(slot_teams) == team) & valid
    indices = np.where(mask)[0]

    if len(indices) > 0:
        nearest = indices[np.argmin(distances[indices])]

        print(
            f"{team}: Slot {nearest}, "
            f"Entfernung {distances[nearest]:.2f} m, "
            f"beobachtete Spieler {len(indices)}"
        )

In [ ]:

# Minimale Entfernung pro Mannschaft
team_mask = np.array(slot_teams)

d_H = np.min(distances[team_mask == "H"])
d_A = np.min(distances[team_mask == "A"])

# Räumlicher Vorteil aus Barcelonas Sicht
proximity_advantage = d_A - d_H

# Normalisierter räumlicher Vorteil
proximity_normalized = (
    (d_A - d_H) / (d_A + d_H + 1e-6)
)

print(f"Barcelona-Distanz: {d_H:.2f} m")
print(f"Athletic-Distanz: {d_A:.2f} m")
print(f"Proximity Advantage: {proximity_advantage:.2f} m")
print(f"Normalisierter Vorteil: {proximity_normalized:.3f}")

In [ ]:
team_mask = np.array(slot_teams)
temporal_records = []

with RAW.open(encoding="utf-8") as file:
    for line in file:
        frame_id, players_raw, ball_raw = line.strip().split(":")
        frame = int(frame_id) - 1

        if frame < 11187:
            continue
        if frame > 11286:
            break

        values = np.fromstring(
            players_raw.replace(";", ",") + "," + ball_raw,
            sep=",",
            dtype=np.float32
        ).reshape(47, 3)

        player_xy = values[:-1, :2]
        ball_xy = values[-1, :2]

        if not np.isfinite(ball_xy).all():
            continue

        valid = np.isfinite(player_xy).all(axis=1)

        distances = np.linalg.norm(
            (player_xy - ball_xy) * pitch_scale,
            axis=1
        )
        distances[~valid] = np.inf

        d_H = distances[team_mask == "H"].min()
        d_A = distances[team_mask == "A"].min()

        if not np.isfinite(d_H) or not np.isfinite(d_A):
            continue

        temporal_records.append({
            "frame": frame,
            "d_H": d_H,
            "d_A": d_A,
            "nearest_team": "H" if d_H < d_A else "A"
        })

df_temporal = pd.DataFrame(temporal_records)

print("Auswertbare Frames:", len(df_temporal))
print("\nNächstes Team pro Frame:")
print(df_temporal["nearest_team"].value_counts())

print("\nMedian der Ballabstände:")
print(df_temporal[["d_H", "d_A"]].median().round(2))

display(df_temporal.head(10).round(2))

In [ ]:

# Ballabstände und Ballgeschwindigkeit zusammenführen
plot_df = df_temporal.merge(
    ball_motion[["frame", "speed_mps"]],
    on="frame",
    how="left"
)

fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 7), sharex=True
)

# 1. Ballabstände
ax1.plot(plot_df["frame"], plot_df["d_H"],
         label="Barcelona", color="blue")
ax1.plot(plot_df["frame"], plot_df["d_A"],
         label="Athletic", color="red")

ax1.set_ylabel("Distance to ball (m)")
ax1.set_title("Temporal ball proximity")
ax1.legend()
ax1.grid(alpha=0.3)

# 2. Ballgeschwindigkeit
ax2.plot(plot_df["frame"], plot_df["speed_mps"],
         color="black")

ax2.set_ylabel("Ball speed (m/s)")
ax2.set_xlabel("Video frame")
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:

contact_records = []

team_mask = np.array(slot_teams)

with RAW.open(encoding="utf-8") as file:
    for line in file:
        frame_id, players_raw, ball_raw = line.strip().split(":")
        frame = int(frame_id) - 1

        if frame < 11187:
            continue
        if frame > 11236:
            break

        values = np.fromstring(
            players_raw.replace(";", ",") + "," + ball_raw,
            sep=",",
            dtype=np.float32
        ).reshape(47, 3)

        player_xy = values[:-1, :2]
        ball_xy = values[-1, :2]

        if not np.isfinite(ball_xy).all():
            continue

        distances = np.linalg.norm(
            (player_xy - ball_xy) * pitch_scale,
            axis=1
        )
        distances[~np.isfinite(player_xy).all(axis=1)] = np.inf

        if not np.isfinite(distances).any():
            continue

        nearest_slot = int(np.argmin(distances))
        nearest_distance = float(distances[nearest_slot])

        contact_records.append({
            "frame": frame,
            "nearest_team": team_mask[nearest_slot],
            "nearest_slot": nearest_slot,
            "distance_m": nearest_distance,
            "within_1m": nearest_distance <= 1.0,
            "within_2m": nearest_distance <= 2.0
        })

df_contacts = pd.DataFrame(contact_records)

print("Frames mit Abstand <= 1 m:",
      df_contacts["within_1m"].sum())

print("Frames mit Abstand <= 2 m:",
      df_contacts["within_2m"].sum())

print("\nNächste Spieler-Slots:")
print(df_contacts["nearest_slot"].value_counts())

print("\nZeitlicher Verlauf (jeder 5. Frame):")
display(df_contacts.iloc[::5].round(2))

In [ ]:

target_slot = 34
records = []

with RAW.open(encoding="utf-8") as file:
    for line in file:
        frame_id, players_raw, ball_raw = line.strip().split(":")
        frame = int(frame_id) - 1

        if frame < 11187:
            continue
        if frame > 11236:
            break

        values = np.fromstring(
            players_raw.replace(";", ",") + "," + ball_raw,
            sep=",",
            dtype=np.float32
        ).reshape(47, 3)

        player = values[target_slot, :2] * pitch_scale
        ball = values[-1, :2] * pitch_scale

        records.append({
            "frame": frame,
            "player_x": player[0],
            "player_y": player[1],
            "ball_x": ball[0],
            "ball_y": ball[1]
        })

df_relative = pd.DataFrame(records)

# Abstand zwischen Ball und Spieler
df_relative["distance_m"] = np.hypot(
    df_relative["ball_x"] - df_relative["player_x"],
    df_relative["ball_y"] - df_relative["player_y"]
)

# Geschwindigkeiten über 5 Frames
k = 5
dt = k / 25

for obj in ["ball", "player"]:
    df_relative[f"{obj}_vx"] = (
        df_relative[f"{obj}_x"].diff(k) / dt
    )
    df_relative[f"{obj}_vy"] = (
        df_relative[f"{obj}_y"].diff(k) / dt
    )
    df_relative[f"{obj}_speed"] = np.hypot(
        df_relative[f"{obj}_vx"],
        df_relative[f"{obj}_vy"]
    )

# Wie schnell ändert sich der gegenseitige Abstand?
df_relative["separation_rate"] = (
    df_relative["distance_m"].diff(k) / dt
)

display(
    df_relative[
        ["frame", "distance_m", "ball_speed",
         "player_speed", "separation_rate"]
    ].iloc[::5].round(2)
)

In [ ]:

start_frame = 11187
end_frame = start_frame + 300  # 12 Sekunden

contact_window = []

with RAW.open(encoding="utf-8") as file:
    for line in file:
        parts = line.strip().split(":")
        frame = int(parts[0]) - 1

        if frame < start_frame:
            continue
        if frame > end_frame:
            break

        values = np.fromstring(
            parts[1].replace(";", ",") + "," + parts[2],
            sep=",",
            dtype=np.float32
        ).reshape(47, 3)

        player_xy = values[:-1, :2]
        ball_xy = values[-1, :2]

        if not np.isfinite(ball_xy).all():
            continue

        distances = np.linalg.norm(
            (player_xy - ball_xy) * pitch_scale,
            axis=1
        )

        distances[~np.isfinite(player_xy).all(axis=1)] = np.inf

        if not np.isfinite(distances).any():
            continue

        nearest = int(np.argmin(distances))

        contact_window.append({
            "frame": frame,
            "team": team_mask[nearest],
            "slot": nearest,
            "distance_m": float(distances[nearest])
        })

df_contact_window = pd.DataFrame(contact_window)

# Kandidaten für Ballkontakte
candidates = df_contact_window[
    df_contact_window["distance_m"] <= 2.0
]

print("Frames insgesamt:", end_frame - start_frame + 1)
print("Frames mit auswertbarem Tracking:", len(df_contact_window))
print("Frames mit möglichem Ballkontakt:", len(candidates))

display(candidates.head(30).round(2))

In [ ]:

# Frames mit räumlicher Ballnähe
contacts = df_contact_window[
    df_contact_window["distance_m"] <= 2.0
].copy().sort_values("frame").reset_index(drop=True)

# Ein neues Kontaktfenster beginnt bei:
# - zeitlicher Lücke
# - Mannschaftswechsel
# - Wechsel des Tracking-Slots
new_episode = (
    contacts["frame"].diff().ne(1)
    | contacts["team"].ne(contacts["team"].shift())
    | contacts["slot"].ne(contacts["slot"].shift())
)

contacts["episode_id"] = new_episode.cumsum()

# Kontaktfenster zusammenfassen
episodes = contacts.groupby(
    "episode_id", as_index=False
).agg(
    team=("team", "first"),
    slot=("slot", "first"),
    start_frame=("frame", "min"),
    end_frame=("frame", "max"),
    n_frames=("frame", "size"),
    min_distance_m=("distance_m", "min")
)

# Dauer in Sekunden
episodes["duration_s"] = episodes["n_frames"] / 25

print("Anzahl Kontaktfenster:", len(episodes))
print("\nKontaktfenster nach Mannschaft:")
print(episodes["team"].value_counts())

display(episodes.round(2))

In [ ]:

events = {
    "Barcelona-Kontakt": 11263,
    "Athletic-Kontakt": 11266,
    "Torwart-Kontakt": 11476
}

k = 3
ball_positions = {}

with RAW.open(encoding="utf-8") as file:
    for line in file:
        frame_id, _, ball_raw = line.strip().split(":")
        frame = int(frame_id) - 1

        if frame < min(events.values()) - k:
            continue
        if frame > max(events.values()) + k:
            break

        ball = np.fromstring(ball_raw, sep=",")
        if ball.size >= 2 and np.isfinite(ball[:2]).all():
            ball_positions[frame] = ball[:2] * pitch_scale

results = []

for event, frame in events.items():
    p_before = ball_positions.get(frame - k)
    p_now = ball_positions.get(frame)
    p_after = ball_positions.get(frame + k)

    if any(p is None for p in [p_before, p_now, p_after]):
        results.append({"event": event, "frame": frame})
        continue

    v_before = (p_now - p_before) * 25 / k
    v_after = (p_after - p_now) * 25 / k

    speed_before = np.linalg.norm(v_before)
    speed_after = np.linalg.norm(v_after)

    if speed_before > 0.2 and speed_after > 0.2:
        cosine = np.dot(v_before, v_after) / (
            speed_before * speed_after
        )
        angle = np.degrees(
            np.arccos(np.clip(cosine, -1, 1))
        )
    else:
        angle = np.nan

    results.append({
        "event": event,
        "frame": frame,
        "speed_before": speed_before,
        "speed_after": speed_after,
        "direction_change_deg": angle
    })

df_direction = pd.DataFrame(results)
display(df_direction.round(2))

In [ ]:

frames_to_show = [11259, 11261, 11263, 11265, 11267, 11270]

cap = cv2.VideoCapture(str(video_path))
fig, axes = plt.subplots(2, 3, figsize=(18, 8))

for ax, frame in zip(axes.flat, frames_to_show):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame)
    success, image = cap.read()

    if success:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        ax.imshow(image)

    ax.set_title(f"Frame {frame} | {frame / 25:.2f} s")
    ax.axis("off")

cap.release()
plt.tight_layout()
plt.show()

In [ ]:

# Videozeitpunkte rund um die Torwartaktion
times_s = [460, 462, 464, 465, 466, 468]
frames_to_show = [int(t * 25) for t in times_s]

cap = cv2.VideoCapture(str(video_path))

fig, axes = plt.subplots(2, 3, figsize=(18, 9))

for ax, frame, t in zip(axes.flat, frames_to_show, times_s):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame)
    success, image = cap.read()

    if success:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        ax.imshow(image)

    ax.set_title(f"Frame {frame} | {t//60}:{t%60:02d}")
    ax.axis("off")

cap.release()
plt.tight_layout()
plt.show()

In [ ]:

records = []

with RAW.open(encoding="utf-8") as file:
    for line in file:
        frame_id, players_raw, ball_raw = line.strip().split(":")
        frame = int(frame_id) - 1

        if frame < 11500:
            continue
        if frame > 11700:
            break

        values = np.fromstring(
            players_raw.replace(";", ",") + "," + ball_raw,
            sep=",",
            dtype=np.float32
        ).reshape(47, 3)

        players_xy = values[:-1, :2]
        ball_xy = values[-1, :2]

        ball_valid = np.isfinite(ball_xy).all()

        d_H, d_A = np.nan, np.nan

        if ball_valid:
            distances = np.linalg.norm(
                (players_xy - ball_xy) * pitch_scale,
                axis=1
            )
            distances[~np.isfinite(players_xy).all(axis=1)] = np.inf

            for team in ["H", "A"]:
                d = distances[team_mask == team].min()
                if np.isfinite(d):
                    if team == "H":
                        d_H = d
                    else:
                        d_A = d

        records.append({
            "frame": frame,
            "ball_valid": bool(ball_valid),
            "d_H": d_H,
            "d_A": d_A
        })

df_goalkeeper = pd.DataFrame(records)

print("Ballverfügbarkeit:",
      f"{df_goalkeeper['ball_valid'].mean():.1%}")

display(df_goalkeeper.iloc[::25].round(2))

In [ ]:

N_FRAMES = 199_319

# Speicher für das vollständige Tracking
tracking = np.full(
    (N_FRAMES, 47, 3),
    np.nan,
    dtype=np.float32
)

seen = np.zeros(N_FRAMES, dtype=bool)

with RAW.open(encoding="utf-8") as file:
    for line in file:

        frame_id, players_raw, ball_raw = line.strip().split(":")
        frame = int(frame_id) - 1

        if not 0 <= frame < N_FRAMES or seen[frame]:
            raise ValueError(f"Ungültiger Frame: {frame}")

        values = np.fromstring(
            players_raw.replace(";", ",") + "," + ball_raw,
            sep=",",
            dtype=np.float32
        )

        if values.size != 47 * 3:
            raise ValueError(f"Ungültige Daten in Frame {frame}")

        tracking[frame] = values.reshape(47, 3)
        seen[frame] = True

assert seen.all(), "Es fehlen Tracking-Frames."

# Beobachtungsqualität
player_valid = np.isfinite(
    tracking[:, :46, :2]
).all(axis=2)

ball_valid = np.isfinite(
    tracking[:, 46, :2]
).all(axis=1)

print("Tracking-Shape:", tracking.shape)
print("Ballverfügbarkeit:", f"{ball_valid.mean():.2%}")

print("Median beobachteter Spieler:")
print("Barcelona:", np.median(player_valid[:, team_mask == "H"].sum(axis=1)))
print("Athletic:", np.median(player_valid[:, team_mask == "A"].sum(axis=1)))

In [ ]:

# Team-Masken
mask_H = team_mask == "H"
mask_A = team_mask == "A"

# Ball-Spieler-Distanzen in Metern für alle Frames
distances = np.hypot(
    (tracking[:, :46, 0] - tracking[:, 46, 0, None]) * 105.3,
    (tracking[:, :46, 1] - tracking[:, 46, 1, None]) * 68.0
)

# Ungültige Spielerpositionen ausschließen
distances[~player_valid] = np.inf

# Frames ohne Ballposition ausschließen
distances[~ball_valid, :] = np.inf

# Minimale Distanz pro Team
d_H = distances[:, mask_H].min(axis=1)
d_A = distances[:, mask_A].min(axis=1)

# Fehlende Werte als NaN darstellen
d_H[~np.isfinite(d_H)] = np.nan
d_A[~np.isfinite(d_A)] = np.nan

# Feature-DataFrame
df_features = pd.DataFrame({
    "frame": np.arange(N_FRAMES),
    "ball_valid": ball_valid,
    "n_H": player_valid[:, mask_H].sum(axis=1),
    "n_A": player_valid[:, mask_A].sum(axis=1),
    "d_H_m": d_H,
    "d_A_m": d_A,
    "advantage_H_m": d_A - d_H
})

print("Feature-Shape:", df_features.shape)
print("Frames mit beiden Distanzen:", df_features[["d_H_m", "d_A_m"]].notna().all(axis=1).sum())

print("\nKontrolle Frame 11187:")
display(df_features.iloc[[11187]].round(2))

In [ ]:

# Trackingqualität pro Videominute
df_quality = df_features.copy()

df_quality["video_minute"] = (
    df_quality["frame"] // (25 * 60)
)

df_quality["both_distances_valid"] = (
    df_quality[["d_H_m", "d_A_m"]]
    .notna().all(axis=1)
)

quality_by_minute = df_quality.groupby(
    "video_minute"
).agg(
    ball_availability=("ball_valid", "mean"),
    distance_availability=("both_distances_valid", "mean"),
    players_H=("n_H", "median"),
    players_A=("n_A", "median")
).reset_index()

# Visualisierung
fig, axes = plt.subplots(
    2, 1, figsize=(14, 8), sharex=True
)

axes[0].plot(
    quality_by_minute["video_minute"],
    quality_by_minute["ball_availability"],
    label="Ball verfügbar"
)
axes[0].plot(
    quality_by_minute["video_minute"],
    quality_by_minute["distance_availability"],
    label="Beide Distanzen verfügbar",
    alpha=0.7
)
axes[0].set_ylabel("Anteil")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(
    quality_by_minute["video_minute"],
    quality_by_minute["players_H"],
    label="Barcelona"
)
axes[1].plot(
    quality_by_minute["video_minute"],
    quality_by_minute["players_A"],
    label="Athletic"
)
axes[1].set_ylabel("Median beobachteter Spieler")
axes[1].set_xlabel("Videominute")
axes[1].legend()
axes[1].grid(alpha=0.3)

# Bekannte Halbzeit-Anstöße markieren
kickoff_starts = df_kickoffs[
    df_kickoffs["type"] == "Kick Off (Start)"
]

for ax in axes:
    for frame in kickoff_starts["video_frame"]:
        ax.axvline(
            frame / (25 * 60),
            color="gray",
            linestyle="--",
            alpha=0.6
        )

plt.tight_layout()
plt.show()

In [ ]:

# Auffällige Minuten als Qualitätsdiagnose
suspicious_minutes = quality_by_minute.loc[
    (quality_by_minute["ball_availability"] >= 0.80)
    & (
        (quality_by_minute["players_H"] < 8)
        | (quality_by_minute["players_A"] < 8)
    )
]

print("Auffällige Videominuten:")
display(suspicious_minutes.round(2))

# Frames mit mehr als 11 erkannten Slots pro Team
extra_players = (
    (df_features["n_H"] > 11)
    | (df_features["n_A"] > 11)
)

print("\nFrames mit mehr als 11 Spieler-Slots:",
      extra_players.sum())

print("Anteil am gesamten Video:",
      f"{extra_players.mean():.2%}")

In [ ]:

tag_records = []

for instance in instances:
    labels = {
        label.findtext("group"): label.findtext("text")
        for label in instance.findall("label")
    }

    tag_records.append({
        "half": labels.get("Half"),
        "type": labels.get("Type"),
        "code": instance.findtext("code"),
        "start_s": float(instance.findtext("start")),
        "end_s": float(instance.findtext("end"))
    })

df_tags = pd.DataFrame(tag_records)

print("Zeitbereiche der Smart Tags pro Halbzeit:")

display(
    df_tags.groupby("half").agg(
        first_event_s=("start_s", "min"),
        last_event_s=("end_s", "max"),
        n_events=("code", "size")
    ).round(2)
)

In [ ]:

# Anstoßzeitpunkte aus den Smart Tags
kickoff_frames = df_kickoffs.loc[
    df_kickoffs["type"] == "Kick Off (Start)",
    "video_frame"
].sort_values().to_numpy()

# Konservative Untersuchungsfenster
frames = df_features["frame"].to_numpy()
analysis_mask = np.zeros(len(df_features), dtype=bool)

for kickoff in kickoff_frames:
    analysis_mask |= (
        (frames >= kickoff + 2 * 60 * 25)
        & (frames < kickoff + 40 * 60 * 25)
    )

# Nur Frames mit auswertbaren Ballabständen
valid_distances = df_features[
    ["d_H_m", "d_A_m"]
].notna().all(axis=1).to_numpy()

df_analysis = df_features.loc[
    analysis_mask & valid_distances
].copy()

print("Untersuchte Frames:", analysis_mask.sum())
print("Frames mit gültigen Ballabständen:", len(df_analysis))

print("\nVerteilung der Ballabstände:")
display(
    df_analysis[
        ["d_H_m", "d_A_m", "advantage_H_m"]
    ].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    ).round(2)
)

# Abstand zum nächstgelegenen Spieler
df_analysis["nearest_distance_m"] = df_analysis[
    ["d_H_m", "d_A_m"]
].min(axis=1)

print("\nAbstand zum nächsten Spieler:")
display(
    df_analysis["nearest_distance_m"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
    .round(2)
)

In [ ]:

# Räumlichen Abstandsvorteil untersuchen
df_spatial = df_analysis.copy()

df_spatial["distance_gap_m"] = (
    df_spatial["d_H_m"] - df_spatial["d_A_m"]
).abs()

# Abstand zum nächsten Spieler kategorisieren
df_spatial["distance_group"] = pd.cut(
    df_spatial["nearest_distance_m"],
    bins=[0, 1, 2, 5, np.inf],
    labels=["0–1 m", "1–2 m", "2–5 m", ">5 m"],
    include_lowest=True,
    right=False
)

# Verteilung auswerten
summary = df_spatial.groupby(
    "distance_group", observed=True
).agg(
    n_frames=("frame", "size"),
    median_gap_m=("distance_gap_m", "median"),
    share_gap_under_1m=(
        "distance_gap_m",
        lambda x: (x < 1).mean()
    )
)

summary["share_frames"] = (
    summary["n_frames"] / len(df_spatial)
)

display(summary.round(3))

In [ ]:

# Räumlich nächstes Team bestimmen
df_stability = df_analysis.sort_values("frame").copy()

df_stability["nearest_team"] = np.where(
    df_stability["d_H_m"] < df_stability["d_A_m"],
    "H",
    np.where(
        df_stability["d_A_m"] < df_stability["d_H_m"],
        "A",
        "TIE"
    )
)

# Neue Phase bei Teamwechsel oder fehlenden Zwischenframes
new_phase = (
    df_stability["nearest_team"].ne(
        df_stability["nearest_team"].shift()
    )
    | df_stability["frame"].diff().ne(1)
)

df_stability["phase_id"] = new_phase.cumsum()

# Zusammenhängende räumliche Phasen
spatial_phases = df_stability.groupby(
    "phase_id", as_index=False
).agg(
    team=("nearest_team", "first"),
    start_frame=("frame", "min"),
    end_frame=("frame", "max"),
    n_frames=("frame", "size")
)

spatial_phases["duration_s"] = (
    spatial_phases["n_frames"] / 25
)

print("Anzahl räumlicher Phasen:", len(spatial_phases))

print("\nPhasendauer in Sekunden:")
display(
    spatial_phases["duration_s"].describe(
        percentiles=[0.10, 0.25, 0.50, 0.75, 0.90]
    ).round(2)
)

print("\nAnteil sehr kurzer Phasen (< 0,2 s):")
print(f"{(spatial_phases['duration_s'] < 0.2).mean():.1%}")

print("\nDie 10 längsten Phasen:")
display(
    spatial_phases.nlargest(
        10, "duration_s"
    ).round(2)
)

In [ ]:

# Zeitlich sortieren
df_stability = df_stability.sort_values(
    "frame"
).reset_index(drop=True)

# Vorherigen Frame und vorherige Mannschaft vergleichen
frame_diff = df_stability["frame"].diff()

team_change = df_stability["nearest_team"].ne(
    df_stability["nearest_team"].shift()
)

tracking_gap = frame_diff > 1

# Ursache für jede neue Phase bestimmen
df_stability["start_reason"] = np.select(
    [
        df_stability.index == 0,
        tracking_gap & team_change,
        tracking_gap,
        team_change
    ],
    [
        "Erste Phase",
        "Trackinglücke + anderes Team",
        "Trackinglücke, gleiches Team",
        "Direkter räumlicher Wechsel"
    ],
    default="Keine neue Phase"
)

# Nur Phasenanfänge betrachten
phase_starts = df_stability[
    df_stability["start_reason"] != "Keine neue Phase"
]

print("Ursachen neuer räumlicher Phasen:")
display(
    phase_starts["start_reason"]
    .value_counts()
    .rename_axis("Ursache")
    .reset_index(name="Anzahl")
)

# Direkte Wechsel bei aufeinanderfolgenden Frames
direct_changes = (
    df_stability["start_reason"]
    == "Direkter räumlicher Wechsel"
).sum()

print("\nDirekte räumliche Teamwechsel:", direct_changes)
print("Räumliche Phasen insgesamt:", len(phase_starts))

In [ ]:

# Alle Frames unserer Untersuchungsfenster behalten
# Einschließlich Frames ohne Balltracking
df_evidence = df_features.loc[
    analysis_mask,
    ["frame", "advantage_H_m"]
].copy()

# Halbzeiten getrennt behandeln
df_evidence["half"] = np.where(
    df_evidence["frame"] < kickoff_frames[1], 1, 2
)

# Zeitlicher Median über 25 Frames (1 Sekunde)
df_evidence["advantage_median_1s"] = (
    df_evidence.groupby("half")["advantage_H_m"]
    .transform(
        lambda x: x.rolling(
            window=25,
            center=True,
            min_periods=10
        ).median()
    )
)

# Anteil gültiger Beobachtungen im Zeitfenster
df_evidence["coverage_1s"] = (
    df_evidence.groupby("half")["advantage_H_m"]
    .transform(
        lambda x: x.notna().astype(float).rolling(
            window=25,
            center=True,
            min_periods=1
        ).mean()
    )
)

print("Frames:", len(df_evidence))

print(
    "Frames mit zeitlicher Evidenz:",
    df_evidence["advantage_median_1s"].notna().sum()
)

print("\nTracking-Abdeckung:")
print(df_evidence["coverage_1s"].describe().round(2))

display(
    df_evidence.dropna(
        subset=["advantage_median_1s"]
    ).head(10).round(2)
)

In [ ]:

# Rohwerte und zeitlichen Median vergleichen
df_compare = df_evidence.copy()

# Räumlich nächstes Team bestimmen
for column, name in [
    ("advantage_H_m", "raw_team"),
    ("advantage_median_1s", "smooth_team")
]:
    values = df_compare[column]

    df_compare[name] = np.where(
        values > 0, "H",
        np.where(values < 0, "A", None)
    )

# Nur Frames, für die beide Verfahren gültige Werte liefern
paired = df_compare.dropna(
    subset=["raw_team", "smooth_team"]
).reset_index(drop=True)

# Nur direkt aufeinanderfolgende Frames derselben Halbzeit
adjacent = (
    paired["frame"].diff().eq(1)
    & paired["half"].eq(paired["half"].shift())
)

# Tatsächliche Wechsel der räumlich nächsten Mannschaft
raw_switches = (
    adjacent
    & paired["raw_team"].ne(paired["raw_team"].shift())
).sum()

smooth_switches = (
    adjacent
    & paired["smooth_team"].ne(paired["smooth_team"].shift())
).sum()

# Abweichungen zwischen beiden Verfahren
different = (
    paired["raw_team"] != paired["smooth_team"]
).mean()

print("Gemeinsam auswertbare Frames:", len(paired))
print("Vergleichbare Frame-Übergänge:", adjacent.sum())

print("\nDirekte räumliche Teamwechsel:")
print("Ohne Median:", raw_switches)
print("Mit 1s-Median:", smooth_switches)

print("\nAbweichende Teamzuordnungen:")
print(f"{different:.2%}")


In [ ]:

# Feature-Daten für die Evaluation speichern
FEATURE_PATH = PROJECT_DIR / "output/features_v0.csv"

FEATURE_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_features.to_csv(FEATURE_PATH, index=False)

print("Features gespeichert:", FEATURE_PATH)
print("Shape:", df_features.shape)
